In [3]:
# GPU-Optimized Multimodal RAG with ColPali + MedGemma - Complete Implementation
# Enhanced for Leishmania Research with Peer Suggestions Applied

import os
import json
import time
import logging
import gc
import math
import hashlib
import threading
from datetime import datetime
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
import faiss

# Set up paths and environment (unified with Code A)
os.environ["HF_HOME"] = "/data4t/hf"
os.environ["HF_HUB_CACHE"] = "/data4t/hf/hub"
os.environ["TRANSFORMERS_CACHE"] = "/data4t/hf/transformers"

# Unified path configuration
PROJECT_DIR = Path("/home/students/Leishmania")
A_RAG_DIR = PROJECT_DIR / "kaggle" / "working" / "rag_knowledge_base"
A_IMG_DIR = A_RAG_DIR / "images"
A_META_DIR = A_RAG_DIR / "metadata"
INDEX_DIR = A_RAG_DIR / "index"
COLQWEN2_DIR = INDEX_DIR / "colqwen2"
PAGE_RENDERS_DIR = A_RAG_DIR / "page_renders"

# Create directories
for d in [A_RAG_DIR, A_IMG_DIR, A_META_DIR, INDEX_DIR, COLQWEN2_DIR, PAGE_RENDERS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# GPU configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEBUG_GPU_SPIN = False  # Disable GPU spinners by default

# Leishmania keywords for intelligent filtering
LEISHMANIA_KEYWORDS = [
    'leishmaniasis', 'leishmania', 'kala-azar', 'visceral leishmaniasis',
    'cutaneous leishmaniasis', 'mucocutaneous leishmaniasis',
    'sandfly', 'phlebotomus', 'lutzomyia', 'amastigotes', 'promastigotes',
    'montenegro test', 'pentavalent antimony', 'amphotericin b',
    'miltefosine', 'chiclero', 'espundia', 'oriental sore'
]

# Load model from manifest to ensure consistency with Code A
MANIFEST_PATH = COLQWEN2_DIR / "manifest.json"
MODEL_NAME_FROM_MANIFEST = None
if MANIFEST_PATH.exists():
    try:
        import json
        with open(MANIFEST_PATH) as f:
            manifest_data = json.load(f)
        MODEL_NAME_FROM_MANIFEST = manifest_data.get("model")
    except Exception:
        pass

logger.info(f"Using device: {device}")
logger.info(f"Project directory: {PROJECT_DIR}")
logger.info(f"RAG directory: {A_RAG_DIR}")

# ============================================================================
# SECTION 1: ColQwen2 Page Renderer and Index Builder (A.5 step)
# ============================================================================

def render_pdf_pages(pdf_path: Path, output_dir: Path, dpi: int = 150) -> List[Dict[str, Any]]:
    """Render PDF pages to images with metadata tracking."""
    try:
        from pdf2image import convert_from_path
        import fitz  # PyMuPDF
        
        doc_id = pdf_path.stem
        doc_output_dir = output_dir / doc_id
        doc_output_dir.mkdir(exist_ok=True)
        
        # Convert PDF to images
        pages = convert_from_path(
            str(pdf_path), 
            dpi=dpi,
            fmt='PNG',
            thread_count=4
        )
        
        page_metadata = []
        for i, page in enumerate(pages):
            page_num = i + 1
            img_filename = f"page_{page_num:04d}.png"
            img_path = doc_output_dir / img_filename
            
            # Optimize image for processing
            if page.mode != 'RGB':
                page = page.convert('RGB')
            if max(page.size) > 2048:
                page.thumbnail((2048, 2048), Image.Resampling.LANCZOS)
                
            page.save(img_path, "PNG", optimize=True)
            
            page_metadata.append({
                "source_file": str(pdf_path),
                "doc_id": doc_id,
                "page": page_num,
                "image_path": str(img_path),
                "width": page.width,
                "height": page.height,
                "row_id": len(page_metadata)  # Stable row ID for indexing
            })
            
        logger.info(f"Rendered {len(pages)} pages from {pdf_path.name}")
        return page_metadata
        
    except Exception as e:
        logger.error(f"Error rendering {pdf_path}: {e}")
        return []

def build_colqwen2_index():
    """Build ColQwen2 index with packed embeddings and FAISS fallback."""
    from transformers import AutoProcessor, AutoModel
    from transformers.utils import is_flash_attn_2_available
    
    MODEL_NAME = MODEL_NAME_FROM_MANIFEST or "vidore/colqwen2-v1.0-hf"
    DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    BATCH_SIZE = 4
    
    logger.info(f"Loading ColQwen2 model: {MODEL_NAME}")
    
    # Load model with proper configuration
    model = AutoModel.from_pretrained(
        MODEL_NAME,
        torch_dtype=DTYPE,
        device_map="auto" if torch.cuda.is_available() else None,
        attn_implementation="eager",  # Stable implementation
        trust_remote_code=True
    )
    
    processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
    
    if not torch.cuda.is_available():
        model = model.to(device)
    
    model.eval()
    
    # Load image catalog
    def load_image_catalog() -> List[Dict[str, Any]]:
        items = []
        
        # Try to load from existing metadata
        if A_META_DIR.exists():
            for p in sorted(A_META_DIR.glob("*_images.json")):
                try:
                    data = json.loads(p.read_text(encoding="utf-8"))
                    for row in data:
                        if row.get("image_path") and Path(row["image_path"]).exists():
                            items.append(row)
                except Exception as e:
                    logger.warning(f"Error loading {p.name}: {e}")
        
        # Fallback: scan page renders directory
        if not items and PAGE_RENDERS_DIR.exists():
            for img_path in sorted(PAGE_RENDERS_DIR.rglob("*.png")):
                items.append({
                    "source_file": None,
                    "doc_id": img_path.parent.name,
                    "page": int(img_path.stem.split('_')[-1]) if '_' in img_path.stem else 1,
                    "image_path": str(img_path),
                    "row_id": len(items)
                })
        
        return items
    
    catalog = load_image_catalog()
    
    if not catalog:
        logger.warning("No images found. Creating demo content...")
        # Create demo content for testing
        demo_img = Image.new('RGB', (400, 300), color='white')
        demo_path = PAGE_RENDERS_DIR / "demo" / "page_0001.png"
        demo_path.parent.mkdir(exist_ok=True)
        demo_img.save(demo_path)
        
        catalog = [{
            "source_file": "demo.pdf",
            "doc_id": "demo",
            "page": 1,
            "image_path": str(demo_path),
            "row_id": 0
        }]
    
    logger.info(f"Processing {len(catalog)} images")
    
    # Encode images in batches
    packed = []
    offsets = []
    avg_vecs = []
    start = 0
    dim = None
    
    all_paths = [item["image_path"] for item in catalog]
    
    for i in range(0, len(all_paths), BATCH_SIZE):
        batch_paths = all_paths[i:i+BATCH_SIZE]
        
        try:
            # Load images
            images = []
            for path in batch_paths:
                with Image.open(path) as img:
                    images.append(img.convert("RGB"))
            
            # Process with ColQwen2
            inputs = processor(images=images, return_tensors="pt")
            if torch.cuda.is_available():
                inputs = {k: v.to(device) for k, v in inputs.items()}
            
            with torch.no_grad():
                outputs = model(**inputs)
                embeddings = outputs.last_hidden_state  # [batch, seq_len, dim]
                
                for j, emb in enumerate(embeddings):
                    # Remove padding if present
                    if 'attention_mask' in inputs:
                        mask = inputs['attention_mask'][j]
                        emb = emb[mask.bool()]
                    
                    # Normalize for cosine similarity
                    emb = F.normalize(emb.float(), dim=-1)
                    
                    if dim is None:
                        dim = emb.shape[-1]
                    
                    seq_len = emb.shape[0]
                    
                    # Store packed embeddings
                    packed.append(emb.to(torch.float16).cpu().numpy())
                    
                    # Compute average embedding
                    avg_emb = F.normalize(emb.mean(dim=0, keepdim=True), dim=-1)[0]
                    avg_vecs.append(avg_emb.to(torch.float32).cpu().numpy())
                    
                    # Store offset
                    offsets.append((start, seq_len))
                    start += seq_len
                    
        except Exception as e:
            logger.error(f"Error processing batch {i//BATCH_SIZE + 1}: {e}")
            continue
            
        if (i // BATCH_SIZE + 1) % 10 == 0:
            logger.info(f"Processed {i + len(batch_paths)}/{len(all_paths)} images")
            
        # Memory cleanup
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
    
    # Save packed embeddings and metadata
    if packed:
        E = np.concatenate(packed, axis=0)
        AVG = np.stack(avg_vecs, axis=0).astype(np.float32)
        OFF = np.asarray(offsets, dtype=np.int64)
        
        # Save files
        E.tofile(COLQWEN2_DIR / "packed_embeddings.f16.npy")
        np.save(COLQWEN2_DIR / "avg_vectors.npy", AVG)
        np.save(COLQWEN2_DIR / "offsets.npy", OFF)
        
        # Save metadata with stable row_id
        with open(COLQWEN2_DIR / "meta.jsonl", "w", encoding="utf-8") as f:
            for row_id, item in enumerate(catalog):
                item_copy = dict(item)
                item_copy["row_id"] = row_id
                f.write(json.dumps(item_copy, ensure_ascii=False) + "\n")
        
        # Build FAISS index
        # Ensure L2-normalized
        row_norms = np.linalg.norm(AVG, axis=1, keepdims=True) + 1e-12
        AVG = AVG / row_norms
        
        index = faiss.IndexFlatIP(AVG.shape[1])
        index.add(AVG)
        faiss.write_index(index, str(INDEX_DIR / "image_avg.faiss"))
        
        # Save manifest
        manifest = {
            "created_at": datetime.now().isoformat(),
            "model": MODEL_NAME,
            "dtype": "float16 (packed), float32 (avg)",
            "dim": int(dim),
            "n_images": len(catalog),
            "n_tokens_total": int(E.shape[0]),
            "files": {
                "packed": str(COLQWEN2_DIR / "packed_embeddings.f16.npy"),
                "offsets": str(COLQWEN2_DIR / "offsets.npy"),
                "avg": str(COLQWEN2_DIR / "avg_vectors.npy"),
                "meta": str(COLQWEN2_DIR / "meta.jsonl"),
                "faiss": str(INDEX_DIR / "image_avg.faiss")
            }
        }
        
        with open(COLQWEN2_DIR / "manifest.json", "w") as f:
            json.dump(manifest, f, indent=2)
        
        logger.info(f"✅ ColQwen2 index built successfully!")
        logger.info(f"   - {len(catalog)} images indexed")
        logger.info(f"   - {E.shape[0]:,} token embeddings")
        logger.info(f"   - Dimension: {dim}")
        
    else:
        logger.error("No embeddings generated!")

# ============================================================================
# SECTION 2: Query-Only ColQwen2 Encoder and Retrieval System
# ============================================================================

class ColQwen2QueryEncoder:
    """Lightweight ColQwen2 encoder for queries only."""
    
    def __init__(self, model_name: str = None):
        # Prefer manifest model id to ensure consistency with Code A
        self.model_name = model_name or MODEL_NAME_FROM_MANIFEST or "vidore/colqwen2-v1.0-hf"
        self._paths = self._load_paths_from_manifest()
        self.device = device
        self.model = None
        self.processor = None
        
        # Lazy loading flags
        self._faiss_idx = None
        self._meta = None
        self._packed_data = None

    def _load_paths_from_manifest(self):
        man_p = COLQWEN2_DIR / "manifest.json"
        ix_p  = INDEX_DIR / "image_index_manifest.json"  # created by A-3
        paths = {}
        if man_p.exists():
            man = json.loads(man_p.read_text())
            # 'files' are absolute in your A-2 manifest
            paths.update(man.get("files", {}))
            # fallback for dim
            paths["dim"] = man.get("dim")
        # optional: use A-3’s index manifest if present
        if ix_p.exists():
            ix = json.loads(ix_p.read_text())
            active = ix.get("active")
            if active and active in ix:
                paths["faiss"] = ix[active]["file"]
                # could also read vectors/meta if you decide to move them
        return paths
        
    def _load_model(self):
        """Lazy load the model."""
        if self.model is None:
            from transformers import ColQwen2ForRetrieval, ColQwen2Processor
            
            logger.info(f"Loading ColQwen2 query encoder: {self.model_name}")

            self.model = ColQwen2ForRetrieval.from_pretrained(
                self.model_name,
                torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
                device_map="auto" if torch.cuda.is_available() else None,
                attn_implementation="eager",
                trust_remote_code=True
            )

            self.processor = ColQwen2Processor.from_pretrained(
                self.model_name,
                trust_remote_code=True
            )
            
            if not torch.cuda.is_available():
                self.model = self.model.to(self.device)
            
            self.model.eval()
    
    def _load_faiss(self):
        """Lazy load FAISS index."""
        if self._faiss_idx is None:
            faiss_path = Path(self._paths.get("faiss") or (INDEX_DIR / "image_avg.faiss"))
            if faiss_path.exists():
                self._faiss_idx = faiss.read_index(str(faiss_path))
                logger.info(f"Loaded FAISS index: {faiss_path}")
            else:
                raise FileNotFoundError(f"FAISS index not found: {faiss_path}")
        return self._faiss_idx
    
    def _load_meta(self):
        """Lazy load metadata."""
        if self._meta is None:
            meta_path = Path(self._paths.get("meta") or (COLQWEN2_DIR / "meta.jsonl"))
            if meta_path.exists():
                self._meta = []
                with open(meta_path, "r", encoding="utf-8") as f:
                    for line in f:
                        self._meta.append(json.loads(line.strip()))
                logger.info(f"Loaded metadata for {len(self._meta)} images")
            else:
                raise FileNotFoundError(f"Metadata not found: {meta_path}")
        return self._meta
    
    def _load_packed(self):
        """Lazy load packed embeddings."""
        if self._packed_data is None:
            packed_path = Path(self._paths.get("packed") or (COLQWEN2_DIR / "packed_embeddings.f16.npy"))
            offsets_path = Path(self._paths.get("offsets") or (COLQWEN2_DIR / "offsets.npy"))
            dim = int(self._paths.get("dim") or 1024)

            if packed_path.exists() and offsets_path.exists():
                arr = np.fromfile(packed_path, dtype=np.float16).astype(np.float32)
                if arr.size % dim != 0:
                    raise ValueError(f"Packed array size {arr.size} not divisible by dim {dim}")
                E = arr.reshape(-1, dim)
                OFF = np.load(offsets_path)
                self._packed_data = (E, OFF, dim)
                logger.info(f"Loaded packed embeddings: {E.shape}")
            else:
                raise FileNotFoundError("Packed embeddings not found")
        
        return self._packed_data
    
    def encode_query_tokens(self, text: str) -> torch.Tensor:
        self._load_model()
        with torch.no_grad():
            inputs = self.processor(text=[text], return_tensors="pt")
            if torch.cuda.is_available():
                inputs = {k: v.to(self.device) for k, v in inputs.items()}
            outs = self.model(**inputs)
            emb = getattr(outs, "embeddings", None)
            if emb is None:
                emb = getattr(outs, "last_hidden_state", None)
            if emb is None:
                raise RuntimeError("ColQwen2 forward returned neither `embeddings` nor `last_hidden_state`")
            Q = F.normalize(emb[0].float(), dim=-1)
        return Q

    
    def encode_query_avg(self, text: str) -> np.ndarray:
        """Encode query text to averaged embedding."""
        Q = self.encode_query_tokens(text)
        q_avg = F.normalize(Q.mean(dim=0, keepdim=True), dim=-1)
        return q_avg.cpu().numpy().astype(np.float32)
    
    def search_avg(self, text: str, k: int = 5) -> List[Dict[str, Any]]:
        """Fast search using averaged embeddings."""
        idx = self._load_faiss()
        meta = self._load_meta()
        
        q = self.encode_query_avg(text)
        D, I = idx.search(q, min(k, len(meta)))
        
        hits = []
        for r in range(I.shape[1]):
            if I[0, r] < len(meta):
                hit = {
                    "rank": r + 1,
                    "score": float(D[0, r]),
                    **meta[I[0, r]]
                }
                hits.append(hit)
        
        return hits
    
    def search_late_interaction(self, text: str, k: int = 5, preselect: int = 64) -> List[Dict[str, Any]]:
        """Precise search using late interaction (MaxSim)."""
        # First, preselect candidates using avg search
        candidates = self.search_avg(text, preselect)
        
        if not candidates:
            return []
        
        # Load packed data
        E, OFF, dim = self._load_packed()
        meta = self._load_meta()
        
        # Encode query
        Q = self.encode_query_tokens(text)  # [Tq, d]
        
        # Compute MaxSim scores
        scores = []
        for candidate in candidates:
            row_id = candidate["row_id"]
            
            if row_id < len(OFF):
                start, length = OFF[row_id]
                if start + length <= E.shape[0]:
                    P = torch.from_numpy(E[start:start+length]).to(Q.device)  # [Tp, d]
                    
                    # MaxSim: max over document tokens, then sum over query tokens
                    sim_matrix = Q @ P.T  # [Tq, Tp]
                    max_sims = torch.max(sim_matrix, dim=1).values  # [Tq]
                    score = max_sims.sum().item()
                    
                    scores.append((score, row_id, candidate))
        
        # Sort by score and return top-k
        scores.sort(key=lambda x: -x[0])
        
        results = []
        for i, (score, row_id, candidate) in enumerate(scores[:k]):
            result = candidate.copy()
            result["rank"] = i + 1
            result["score"] = float(score)
            results.append(result)
        
        return results

# Initialize global query encoder
query_encoder = ColQwen2QueryEncoder()

# ============================================================================
# SECTION 3: MedGemma Answer Generation
# ============================================================================

class GPUOptimizedMedGemma:
    """GPU-optimized MedGemma for medical answer generation."""
    
    def __init__(self, model_id: str = "google/medgemma-7b-it"):
        self.model_id = model_id
        self.device = device
        self.model = None
        self.processor = None
        self.generation_counter = 0
        
    def _load_model(self):
        """Lazy load MedGemma model."""
        if self.model is None:
            from transformers import AutoProcessor, AutoModelForImageTextToText
            
            logger.info(f"Loading MedGemma: {self.model_id}")
            
            try:
                self.processor = AutoProcessor.from_pretrained(
                    self.model_id,
                    trust_remote_code=True
                )
                
                if torch.cuda.is_available():
                    self.model = AutoModelForImageTextToText.from_pretrained(
                        self.model_id,
                        torch_dtype=torch.bfloat16,
                        device_map="auto",
                        trust_remote_code=True,
                        attn_implementation="eager"
                    )
                else:
                    self.model = AutoModelForImageTextToText.from_pretrained(
                        self.model_id,
                        torch_dtype=torch.float32,
                        trust_remote_code=True
                    )
                    self.model = self.model.to(self.device)
                
                self.model.eval()
                
                # Configure tokenizer
                if self.processor.tokenizer.pad_token is None:
                    self.processor.tokenizer.pad_token = self.processor.tokenizer.eos_token
                
                logger.info(f"✅ MedGemma loaded on {self.device}")
                
            except Exception as e:
                logger.error(f"Failed to load MedGemma: {e}")
                raise
    
    def generate_answer(self, query: str, images: List[Image.Image]) -> str:
        """Generate medical answer from query and images."""
        self._load_model()
        
        try:
            # Limit images for memory efficiency
            max_images = 3
            processed_images = images[:max_images] if images else []
            
            logger.info(f"Generating answer with {len(processed_images)} images")
            
            # Prepare messages
            if processed_images:
                messages = [
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": query}
                        ] + [{"type": "image"} for _ in processed_images]
                    }
                ]
            else:
                messages = [{"role": "user", "content": query}]
            
            # Apply chat template
            if hasattr(self.processor, 'apply_chat_template'):
                prompt = self.processor.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True
                )
            else:
                prompt = query
            
            # Process inputs
            inputs = self.processor(
                text=prompt,
                images=processed_images if processed_images else None,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=2048
            )
            
            # Move to device
            for key in inputs:
                if isinstance(inputs[key], torch.Tensor):
                    inputs[key] = inputs[key].to(self.device, non_blocking=True)
            
            # Generate
            with torch.inference_mode():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=256,
                    do_sample=False,
                    temperature=0.7,
                    pad_token_id=self.processor.tokenizer.pad_token_id,
                    eos_token_id=self.processor.tokenizer.eos_token_id,
                    use_cache=False
                )
            
            # Decode response
            if hasattr(self.processor, 'apply_chat_template'):
                full_response = self.processor.tokenizer.decode(outputs[0], skip_special_tokens=True)
                
                # Extract model response
                if "<start_of_turn>model" in full_response:
                    response = full_response.split("<start_of_turn>model")[-1].strip()
                else:
                    input_length = inputs['input_ids'].shape[1]
                    generated_tokens = outputs[0][input_length:]
                    response = self.processor.tokenizer.decode(
                        generated_tokens, 
                        skip_special_tokens=True
                    ).strip()
            else:
                input_length = inputs['input_ids'].shape[1]
                generated_tokens = outputs[0][input_length:]
                response = self.processor.tokenizer.decode(
                    generated_tokens,
                    skip_special_tokens=True
                ).strip()
            
            # Cleanup
            del inputs, outputs
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            
            return response or "Generated response is empty."
            
        except Exception as e:
            logger.error(f"MedGemma generation failed: {e}")
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            return f"Generation error: {str(e)}"

# Initialize MedGemma
medgemma = GPUOptimizedMedGemma()

# ============================================================================
# SECTION 4: Multimodal Query System
# ============================================================================

def is_leishmania_related(text: str) -> bool:
    """Check if text is related to Leishmania."""
    text_lower = text.lower()
    return any(keyword in text_lower for keyword in LEISHMANIA_KEYWORDS)

def smart_query_system(query: str, query_images: Optional[List[str]] = None, 
                      top_k: int = 5, prioritize_leishmania: bool = True,
                      use_late_interaction: bool = True) -> Dict[str, Any]:
    """Enhanced query system using precomputed ColQwen2 embeddings."""
    
    start_time = time.time()
    query_images = query_images or []
    is_leishmania_query = is_leishmania_related(query)
    
    try:
        # Stage 1: Retrieval using precomputed embeddings
        logger.info(f"Retrieving top {top_k} documents for: '{query[:50]}...'")
        
        if use_late_interaction and is_leishmania_query:
            # Use precise late interaction for Leishmania queries
            hits = query_encoder.search_late_interaction(query, k=top_k)
        else:
            # Use fast averaged search for general queries
            hits = query_encoder.search_avg(query, k=top_k)
        
        if not hits:
            logger.warning("No documents retrieved")
            return {
                "text": "No relevant documents found for your query.",
                "images": [],
                "metadata": {
                    "query": query,
                    "processing_time": time.time() - start_time,
                    "sources_count": 0
                }
            }
        
        # Stage 2: Load retrieved images
        retrieved_images = []
        valid_hits = []
        
        for hit in hits:
            img_path = hit.get("image_path")
            if img_path and Path(img_path).exists():
                try:
                    img = Image.open(img_path).convert("RGB")
                    retrieved_images.append(img)
                    valid_hits.append(hit)
                except Exception as e:
                    logger.warning(f"Failed to load image {img_path}: {e}")
        
        # Add query images if provided
        all_images = []
        if query_images:
            for img_path in query_images:
                if Path(img_path).exists():
                    try:
                        img = Image.open(img_path).convert("RGB")
                        all_images.append(img)
                    except Exception as e:
                        logger.warning(f"Failed to load query image {img_path}: {e}")
        
        all_images.extend(retrieved_images)
        
        # Stage 3: Generate answer
        logger.info(f"Generating answer with {len(all_images)} images")
        answer = medgemma.generate_answer(query, all_images)
        
        # Format response
        processing_time = time.time() - start_time
        
        # Create metadata
        leishmania_sources = sum(1 for hit in valid_hits 
                               if is_leishmania_related(hit.get("doc_id", "")))
        
        source_info = f"\n\n📄 Sources: {len(valid_hits)} pages retrieved"
        if leishmania_sources > 0:
            source_info += f" ({leishmania_sources} Leishmania-specific)"
        source_info += f"\n⚡ Processing time: {processing_time:.2f}s"
        
        response_images = [
            {
                "path": hit.get("image_path", ""),
                "metadata": hit,
                "relevance_rank": hit.get("rank", i + 1)
            }
            for i, hit in enumerate(valid_hits)
        ]
        
        metadata = {
            "query": query,
            "query_images": query_images,
            "processing_time": processing_time,
            "leishmania_related": is_leishmania_query,
            "sources_count": len(valid_hits),
            "leishmania_sources": leishmania_sources,
            "retrieval_method": "late_interaction" if use_late_interaction and is_leishmania_query else "averaged"
        }
        
        return {
            "text": answer + source_info,
            "images": response_images,
            "metadata": metadata
        }
        
    except Exception as e:
        logger.error(f"Error in smart_query_system: {e}", exc_info=True)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        return {
            "text": f"An error occurred: {str(e)}",
            "images": [],
            "metadata": {
                "query": query,
                "error": str(e),
                "processing_time": time.time() - start_time
            }
        }

# ============================================================================
# SECTION 5: Response Display and Management
# ============================================================================

def display_multimodal_response(result: Dict[str, Any]):
    """Display a multimodal response with proper formatting."""
    print("\n" + "="*70)
    print("           MULTIMODAL RESPONSE")
    print("="*70)
    
    # Display text response
    print(f"💬 Text Response:")
    print(f"{result.get('text', 'No text response available.')}")
    
    # Display images if available
    if result.get('images'):
        print(f"\n🖼️ Visual Evidence ({len(result['images'])} images):")
        print("-" * 50)
        for i, img_info in enumerate(result['images'], 1):
            print(f"  📸 Image {i} (Rank {img_info.get('relevance_rank', 'N/A')}): {os.path.basename(img_info.get('path', ''))}")
            meta = img_info.get('metadata', {})
            print(f"     Source: {meta.get('doc_id', 'unknown')} | Page: {meta.get('page', 'unknown')}")
    
    # Display metadata
    metadata = result.get('metadata', {})
    if metadata:
        print(f"\n📊 Processing Metadata:")
        print(f"   - Query: '{metadata.get('query', 'N/A')}'")
        if metadata.get('query_images'): 
            print(f"   - Query Images: {len(metadata.get('query_images', []))}")
        print(f"   - Processing time: {metadata.get('processing_time', 0):.2f}s")
        print(f"   - Leishmania-related: {metadata.get('leishmania_related', False)}")
        print(f"   - Retrieval method: {metadata.get('retrieval_method', 'unknown')}")
        print(f"   - Sources: {metadata.get('sources_count', 0)} ({metadata.get('leishmania_sources', 0)} Leishmania-specific)")
    print("="*70)

def save_multimodal_response(result: Dict[str, Any], output_dir: Path = None) -> str:
    """Save a multimodal response to files."""
    if output_dir is None:
        output_dir = A_RAG_DIR / "output"
    
    output_dir.mkdir(parents=True, exist_ok=True)
    timestamp = int(time.time())
    
    try:
        # Save text response and metadata
        response_file = output_dir / f"response_{timestamp}.json"
        with open(response_file, 'w', encoding='utf-8') as f:
            json.dump(result, f, indent=2, ensure_ascii=False)
        
        # Copy relevant images
        if result.get('images'):
            img_dir = output_dir / f"response_images_{timestamp}"
            img_dir.mkdir(exist_ok=True)
            
            import shutil
            for img_info in result['images']:
                src_path = Path(img_info['path'])
                if src_path.exists():
                    dst_path = img_dir / src_path.name
                    shutil.copy2(src_path, dst_path)
                    
            logger.info(f"✅ Response and {len(result['images'])} images saved to {output_dir}")
        else:
            logger.info(f"✅ Response saved to {response_file}")
            
        return str(response_file)
        
    except Exception as e:
        logger.error(f"Error saving response: {e}")
        return None

# ============================================================================
# SECTION 6: Testing and Validation System
# ============================================================================

def run_system_tests():
    """Run comprehensive system tests."""
    print("\n" + "="*60)
    print("     GPU-OPTIMIZED MULTIMODAL RAG SYSTEM - TEST SUITE")
    print("="*60 + "\n")
    
    # Check if index exists
    if not (COLQWEN2_DIR / "manifest.json").exists():
        print("⚠️ ColQwen2 index not found. Building index first...")
        build_colqwen2_index()
        print("✅ Index built successfully!")
    
    # Create test image if needed
    test_img_path = A_RAG_DIR / "test_lesion.png"
    if not test_img_path.exists():
        try:
            from PIL import Image, ImageDraw
            img = Image.new('RGB', (300, 200), color='pink')
            draw = ImageDraw.Draw(img)
            draw.ellipse((100, 50, 200, 150), fill='red', outline='darkred')
            draw.text((10, 10), "Test Skin Lesion", fill="black")
            img.save(test_img_path)
            print(f"🖼️ Created test image: {test_img_path}")
        except Exception as e:
            print(f"Could not create test image: {e}")
            test_img_path = None
    
    # Test cases
    test_cases = [
        {
            "description": "Leishmania Text-Only Query",
            "query": "What are the clinical features of cutaneous leishmaniasis?",
            "query_images": None,
            "expected_leishmania": True
        },
        {
            "description": "General Medical Query",
            "query": "What are the symptoms of malaria?",
            "query_images": None,
            "expected_leishmania": False
        },
        {
            "description": "Multimodal Leishmania Query",
            "query": "Analyze this skin lesion for leishmaniasis signs.",
            "query_images": [str(test_img_path)] if test_img_path and test_img_path.exists() else None,
            "expected_leishmania": True
        }
    ]
    
    results = []
    
    for i, case in enumerate(test_cases, 1):
        print(f"\n--- Test Case {i}: {case['description']} ---")
        
        if case.get("query_images") and not case["query_images"][0]:
            print("⚠️ SKIPPING: Test image not available")
            continue
        
        try:
            start_time = time.time()
            
            result = smart_query_system(
                query=case['query'],
                query_images=case['query_images'],
                top_k=3
            )
            
            test_time = time.time() - start_time
            
            # Validate results
            success = True
            issues = []
            
            if not result.get('text'):
                success = False
                issues.append("No text response generated")
            
            metadata = result.get('metadata', {})
            if metadata.get('leishmania_related') != case['expected_leishmania']:
                issues.append(f"Leishmania detection mismatch: expected {case['expected_leishmania']}, got {metadata.get('leishmania_related')}")
            
            if metadata.get('sources_count', 0) == 0:
                issues.append("No sources retrieved")
            
            print(f"🔍 Query: '{case['query']}'")
            if case.get('query_images'):
                print(f"📷 Images: {len(case['query_images'])}")
            
            print(f"⏱️ Processing time: {test_time:.2f}s")
            print(f"📊 Sources: {metadata.get('sources_count', 0)}")
            print(f"🦠 Leishmania detected: {metadata.get('leishmania_related', False)}")
            print(f"🔍 Retrieval method: {metadata.get('retrieval_method', 'unknown')}")
            
            if success:
                print("✅ TEST PASSED")
            else:
                print("❌ TEST FAILED:")
                for issue in issues:
                    print(f"   - {issue}")
            
            results.append({
                'case': case['description'],
                'success': success,
                'time': test_time,
                'issues': issues
            })
            
        except Exception as e:
            print(f"❌ TEST ERROR: {e}")
            results.append({
                'case': case['description'],
                'success': False,
                'time': 0,
                'issues': [str(e)]
            })
        
        print("-" * 50)
    
    # Summary
    passed = sum(1 for r in results if r['success'])
    total = len(results)
    avg_time = sum(r['time'] for r in results) / len(results) if results else 0
    
    print(f"\n📊 TEST SUMMARY:")
    print(f"   Passed: {passed}/{total}")
    print(f"   Average processing time: {avg_time:.2f}s")
    
    if passed == total:
        print("🎉 ALL TESTS PASSED!")
    else:
        print("⚠️ Some tests failed. Check the issues above.")
    
    return results

# ============================================================================
# SECTION 7: Interactive System
# ============================================================================

def interactive_leishmania_rag():
    """Interactive multimodal RAG system."""
    print("\n" + "="*70)
    print("     INTERACTIVE MULTIMODAL LEISHMANIA RAG SYSTEM")
    print("="*70)
    print("🦠 Specialized for Leishmania research")
    print("🚀 GPU-accelerated with ColQwen2 + MedGemma")
    print("🖼️ Full multimodal support (text + images)")
    print("💡 Type 'help' for commands, 'quit' to exit")
    print("="*70 + "\n")
    
    # Check system readiness
    if not (COLQWEN2_DIR / "manifest.json").exists():
        print("⚠️ System not initialized. Building index...")
        build_colqwen2_index()
        print("✅ System ready!")
    
    while True:
        try:
            query = input("🔍 Your question (or 'help'/'quit'): ").strip()
            
            if not query:
                continue
                
            if query.lower() in ['quit', 'exit', 'q']:
                print("👋 Thank you for using the Multimodal Leishmania RAG system!")
                break
            
            if query.lower() == 'help':
                print("\n📚 Available commands:")
                print("  - Ask any medical question")
                print("  - After typing a question, you can add image paths")
                print("  - 'stats': Show system statistics")
                print("  - 'test': Run the test suite")
                print("  - 'rebuild': Rebuild the ColQwen2 index")
                print("  - 'quit': Exit the system")
                continue
            
            if query.lower() == 'stats':
                try:
                    with open(COLQWEN2_DIR / "manifest.json") as f:
                        manifest = json.load(f)
                    
                    print(f"\n📊 System Statistics:")
                    print(f"  - Indexed images: {manifest.get('n_images', 0):,}")
                    print(f"  - Token embeddings: {manifest.get('n_tokens_total', 0):,}")
                    print(f"  - Model: {manifest.get('model', 'unknown')}")
                    print(f"  - Embedding dimension: {manifest.get('dim', 'unknown')}")
                    print(f"  - Created: {manifest.get('created_at', 'unknown')}")
                except Exception as e:
                    print(f"❌ Could not load statistics: {e}")
                continue
            
            if query.lower() == 'test':
                run_system_tests()
                continue
            
            if query.lower() == 'rebuild':
                confirm = input("⚠️ Rebuild index? This will take time (y/n): ").strip().lower()
                if confirm == 'y':
                    build_colqwen2_index()
                    print("✅ Index rebuilt successfully!")
                continue
            
            # Handle multimodal input
            image_input = input("🖼️ Add image paths (optional, comma-separated): ").strip()
            query_images = []
            
            if image_input:
                for path in image_input.split(','):
                    p = Path(path.strip())
                    if p.exists() and p.is_file():
                        query_images.append(str(p))
                        print(f"  ✅ Added image: {p.name}")
                    else:
                        print(f"  ❌ Image not found: {p}")
            
            # Process query
            print(f"\n⏳ Processing query...")
            
            is_leish_query = is_leishmania_related(query)
            if is_leish_query:
                print("🦠 Leishmania-related query detected - using precise retrieval")
            
            result = smart_query_system(
                query=query,
                query_images=query_images,
                prioritize_leishmania=is_leish_query,
                use_late_interaction=is_leish_query
            )
            
            display_multimodal_response(result)
            
            # Offer to save
            save_option = input("\n💾 Save this response? (y/n): ").strip().lower()
            if save_option == 'y':
                saved_path = save_multimodal_response(result)
                if saved_path:
                    print(f"✅ Response saved to: {saved_path}")
                else:
                    print("❌ Failed to save response")
            
        except KeyboardInterrupt:
            print("\n👋 Exiting...")
            break
        except Exception as e:
            print(f"❌ An error occurred: {e}")
            logger.debug("Full error:", exc_info=True)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            continue

# ============================================================================
# SECTION 8: Utility Functions and System Management
# ============================================================================

def cleanup_gpu_memory():
    """Clean up GPU memory and optimize for next operations."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        gc.collect()
        
        memory_allocated = torch.cuda.memory_allocated(0) / (1024**3)
        memory_cached = torch.cuda.memory_reserved(0) / (1024**3)
        
        logger.info(f"GPU memory cleaned - Allocated: {memory_allocated:.2f}GB, Cached: {memory_cached:.2f}GB")
        return {"allocated_gb": memory_allocated, "cached_gb": memory_cached}
    return None

def show_system_summary():
    """Display comprehensive system summary."""
    print("\n" + "="*60)
    print("           SYSTEM SUMMARY")
    print("="*60)
    
    # Check index status
    if (COLQWEN2_DIR / "manifest.json").exists():
        try:
            with open(COLQWEN2_DIR / "manifest.json") as f:
                manifest = json.load(f)
            print(f"📊 Index: {manifest.get('n_images', 0):,} images, {manifest.get('n_tokens_total', 0):,} tokens")
        except Exception:
            print("📊 Index: Present but could not read manifest")
    else:
        print("📊 Index: Not built - run build_colqwen2_index() first")
    
    # GPU info
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        mem_total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        mem_allocated = torch.cuda.memory_allocated(0) / (1024**3)
        print(f"🚀 GPU: {gpu_name} ({mem_total:.1f}GB total, {mem_allocated:.2f}GB allocated)")
    else:
        print("❌ GPU: Not available (using CPU)")
    
    print(f"🤖 Models: ColQwen2 (retrieval), MedGemma (generation)")
    print(f"🖼️ Multimodal: ✅ Input (text+image), ✅ Output (text+image)")
    print(f"🦠 Specialization: Leishmania research with intelligent filtering")
    print("="*60)

def batch_process_queries(queries: List[Dict[str, Any]], output_dir: Path = None) -> List[Dict[str, Any]]:
    """Process multiple queries in batch."""
    if output_dir is None:
        output_dir = A_RAG_DIR / "batch_output"
    
    output_dir.mkdir(parents=True, exist_ok=True)
    logger.info(f"Starting batch processing of {len(queries)} queries...")
    
    results = []
    
    for i, query_data in enumerate(queries, 1):
        text_query = query_data.get("query")
        image_paths = query_data.get("query_images", [])
        
        if not text_query:
            logger.warning(f"Skipping query {i} - no text provided")
            continue
        
        logger.info(f"Processing batch query {i}/{len(queries)}: '{text_query[:50]}...'")
        
        try:
            result = smart_query_system(
                query=text_query,
                query_images=image_paths,
                prioritize_leishmania=is_leishmania_related(text_query)
            )
            
            # Save individual result
            save_multimodal_response(result, output_dir)
            results.append(result)
            
        except Exception as e:
            logger.error(f"Error processing query {i}: {e}")
            results.append({
                "text": f"Error: {str(e)}",
                "images": [],
                "metadata": {"query": text_query, "error": str(e)}
            })
    
    logger.info(f"✅ Batch processing complete - {len(results)} results")
    return results

# ============================================================================
# SECTION 9: Main Entry Points and Initialization
# ============================================================================

def initialize_system():
    """Initialize the complete RAG system."""
    print("🚀 Initializing GPU-Optimized Multimodal RAG System...")
    
    # Check for existing index
    if not (COLQWEN2_DIR / "manifest.json").exists():
        print("📚 No existing index found. Building ColQwen2 index...")
        build_colqwen2_index()
        print("✅ Index built successfully!")
    else:
        print("✅ Using existing ColQwen2 index")
    
    # Load models lazily (they'll be loaded on first use)
    print("🤖 Models configured for lazy loading")
    
    show_system_summary()
    
    print("\n🎉 System initialization complete!")
    print("💡 Available functions:")
    print("  - interactive_leishmania_rag() - Start interactive mode")
    print("  - run_system_tests() - Run comprehensive tests")
    print("  - smart_query_system(query, images) - Direct query processing")
    print("  - build_colqwen2_index() - Rebuild the search index")

# Auto-run initialization if this is the main execution
if __name__ == "__main__":
    initialize_system()
else:
    # When imported as module, show quick status
    show_system_summary()
    print("\n💡 Run initialize_system() to set up the complete RAG pipeline")
    print("💡 Run interactive_leishmania_rag() to start the interactive interface")

2025-08-10 00:18:57,123 - INFO - Using device: cuda
2025-08-10 00:18:57,123 - INFO - Project directory: /home/students/Leishmania
2025-08-10 00:18:57,124 - INFO - RAG directory: /home/students/Leishmania/kaggle/working/rag_knowledge_base


🚀 Initializing GPU-Optimized Multimodal RAG System...
✅ Using existing ColQwen2 index
🤖 Models configured for lazy loading

           SYSTEM SUMMARY
📊 Index: 2,481 images, 1,060,240 tokens
🚀 GPU: NVIDIA GeForce RTX 4090 (23.6GB total, 0.00GB allocated)
🤖 Models: ColQwen2 (retrieval), MedGemma (generation)
🖼️ Multimodal: ✅ Input (text+image), ✅ Output (text+image)
🦠 Specialization: Leishmania research with intelligent filtering

🎉 System initialization complete!
💡 Available functions:
  - interactive_leishmania_rag() - Start interactive mode
  - run_system_tests() - Run comprehensive tests
  - smart_query_system(query, images) - Direct query processing
  - build_colqwen2_index() - Rebuild the search index


In [ ]:
query_encoder = ColQwen2QueryEncoder()
print(len(query_encoder._load_meta()))           # should print your image count (e.g., 2481)
print(query_encoder.search_avg("leishmania", 3)) # should return hits, not crash

2025-08-10 00:19:03,187 - INFO - Loaded metadata for 2481 images
2025-08-10 00:19:03,191 - INFO - Loaded FAISS index: /home/students/Leishmania/kaggle/working/rag_knowledge_base/index/image_avg.faiss
/home/students/Leishmania/.venv/lib/python3.11/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


2481


2025-08-10 00:19:05,269 - INFO - Loading ColQwen2 query encoder: vidore/colqwen2-v1.0-hf
2025-08-10 00:19:06,185 - INFO - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


[{'rank': 1, 'score': 0.45784589648246765, 'source_file': '2-cases-Two cases of cutaneous leishmaniasis in Malawi.pdf', 'doc_id': '2-cases-Two cases of cutaneous leishmaniasis in Malawi', 'page': 1, 'xref': 165, 'image_path': 'kaggle/working/rag_knowledge_base/images/2-cases-Two cases of cutaneous leishmaniasis in Malawi/p0001_xref165.png', 'width': 2400, 'height': 109, 'row_id': 1285}, {'rank': 2, 'score': 0.44953927397727966, 'source_file': '2-cases-Two cases of cutaneous leishmaniasis in Malawi.pdf', 'doc_id': '2-cases-Two cases of cutaneous leishmaniasis in Malawi', 'page': 2, 'xref': 58, 'image_path': 'kaggle/working/rag_knowledge_base/images/2-cases-Two cases of cutaneous leishmaniasis in Malawi/p0002_xref58.png', 'width': 2400, 'height': 109, 'row_id': 1332}, {'rank': 3, 'score': 0.43183720111846924, 'source_file': '2-cases-Two cases of cutaneous leishmaniasis in Malawi.pdf', 'doc_id': '2-cases-Two cases of cutaneous leishmaniasis in Malawi', 'page': 2, 'xref': 22, 'image_path':

generating answer

In [ ]:
import json
import os
from pathlib import Path
from typing import Dict, List, Optional, Any
from tqdm import tqdm
import time
from datetime import datetime

class RAGAnswerGenerator:
    """
    RAG Answer Generator - Generates answers for evaluation questions using the multimodal RAG system.
    This class automates the process of generating answers without performing any evaluation or scoring.
    """
    
    def __init__(self):
        """
        Initialize RAGAnswerGenerator using the available components from cell 18.
        This version works with the function-based approach used in the notebook.
        """
        self.evaluation_questions = []
        self.chunk_data_map = {}
        self.IMAGE_DIR = Path("/home/students/Leishmania/kaggle/working/rag_knowledge_base/images")

        # Auto-load evaluation questions and chunk data
        self._load_evaluation_questions()
        self._load_chunk_data()
        
        print(f"✅ RAGAnswerGenerator initialized with {len(self.evaluation_questions)} questions")
        print(f"📂 Available chunks: {len(self.chunk_data_map)}")
        print(f"🖼️ Image directory: {self.IMAGE_DIR}")
    
    def _load_evaluation_questions(self):
        """Load evaluation questions from the standard location"""
        question_file_path = Path("/home/students/Leishmania/kaggle/working/rag_knowledge_base/evaluation_question_set.json")

        try:
            if question_file_path.exists():
                with open(question_file_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    self.evaluation_questions = data.get('detailed_questions', [])
                print(f"📋 Loaded {len(self.evaluation_questions)} evaluation questions")
            else:
                print(f"⚠️ Question file not found at {question_file_path}")
                self.evaluation_questions = []
        except Exception as e:
            print(f"❌ Error loading evaluation questions: {e}")
            self.evaluation_questions = []
    
    def _load_chunk_data(self):
        """Load and consolidate all chunk data for contextual image finding"""
        chunk_dir = Path("/home/students/Leishmania/kaggle/working/rag_knowledge_base/chunks")
        
        try:
            if chunk_dir.exists():
                for chunk_file in chunk_dir.glob("*.json"):
                    with open(chunk_file, 'r', encoding='utf-8') as f:
                        chunks = json.load(f)
                        if isinstance(chunks, list):
                            for chunk in chunks:
                                if 'chunk_id' in chunk:
                                    self.chunk_data_map[chunk['chunk_id']] = chunk
                        elif isinstance(chunks, dict) and 'chunk_id' in chunks:
                            self.chunk_data_map[chunks['chunk_id']] = chunks
                
                print(f"📚 Loaded chunk data: {len(self.chunk_data_map)} chunks indexed")
            else:
                print(f"⚠️ Chunk directory not found at {chunk_dir}")
        except Exception as e:
            print(f"❌ Error loading chunk data: {e}")
    
    def find_contextual_image_for_question(self, question_data: Dict) -> Optional[str]:
        """
        Find contextual image for a question using source_chunk_id.
        
        Args:
            question_data: Dictionary containing question information with source_chunk_id
            
        Returns:
            Optional[str]: Full path to contextual image if found, None otherwise
        """
        try:
            source_chunk_id = question_data.get('source_chunk_id')
            if not source_chunk_id:
                return None
            
            # Look up chunk in our chunk data map
            chunk_data = self.chunk_data_map.get(source_chunk_id)
            if not chunk_data:
                return None
            
            # Check for associated images
            associated_images = chunk_data.get('associated_images', [])
            if not associated_images:
                return None
            
            # Try to find the first available image with common extensions
            for image_id in associated_images:
                for ext in ['.png', '.jpg', '.jpeg']:
                    image_path = self.IMAGE_DIR / f"{image_id}{ext}"
                    if image_path.exists():
                        return str(image_path)
            
            return None
            
        except Exception as e:
            print(f"⚠️ Error finding contextual image for question {question_data.get('question', '')[:50]}...: {e}")
            return None
    
    def answer_query(self, query: str, image_paths: Optional[List[str]] = None):
        """
        Use the smart_query_system function from cell 18 to answer queries.
        This replaces the rag_system.answer_query() call.
        """
        try:
            # Check if smart_query_system function is available
            if 'smart_query_system' in globals():
                return smart_query_system(query=query, query_images=image_paths)
            else:
                return {
                    'text': 'Error: smart_query_system function not available. Please run cell 18 first.',
                    'images': [],
                    'metadata': {'error': 'smart_query_system not found'}
                }
        except Exception as e:
            return {
                'text': f'Error calling smart_query_system: {str(e)}',
                'images': [],
                'metadata': {'error': str(e)}
            }
    
    def display_response(self, response: Dict[str, Any]):
        """
        Display the RAG response using the display_multimodal_response function from cell 18.
        """
        try:
            if 'display_multimodal_response' in globals():
                display_multimodal_response(response)
            else:
                # Fallback display
                print("💬 RAG Response:")
                print(f"Text: {response.get('text', 'No text response')}")
                if response.get('images'):
                    print(f"Images: {len(response['images'])} retrieved")
                if response.get('metadata'):
                    print(f"Metadata: {response['metadata']}")
        except Exception as e:
            print(f"Error displaying response: {e}")
            print(f"Response: {response}")
    
    def run_generation(self, num_questions_to_run: int = 50):
        """
        Run RAG answer generation for specified number of questions.
        
        Args:
            num_questions_to_run: Number of questions to process (default: 50)
        """
        print(f"\n🚀 Starting RAG Answer Generation for {num_questions_to_run} questions")
        print("="*70)
        
        # Check if required functions are available
        if 'smart_query_system' not in globals():
            print("❌ smart_query_system function not available. Please run cell 18 first.")
            return
        
        if not self.evaluation_questions:
            print("❌ No evaluation questions available. Cannot proceed.")
            return
        
        # Limit to available questions
        questions_to_process = self.evaluation_questions[:num_questions_to_run]
        results = []
        
        # Process questions with progress bar
        for i, question_data in enumerate(tqdm(questions_to_process, desc="Generating answers")):
            try:
                question_text = question_data.get('question', '')
                question_id = question_data.get('question_id', f'q_{i}')
                
                print(f"\n📝 Question {i+1}/{len(questions_to_process)} (ID: {question_id})")
                print(f"❓ {question_text[:100]}{'...' if len(question_text) > 100 else ''}")
                
                # Find contextual image if available
                context_image_path = self.find_contextual_image_for_question(question_data)
                if context_image_path:
                    print(f"🖼️ Found contextual image: {os.path.basename(context_image_path)}")
                    image_paths = [context_image_path]
                else:
                    print("🔍 No contextual image found")
                    image_paths = None
                
                # Generate answer using smart_query_system
                print("🤖 Generating RAG answer...")
                start_time = time.time()
                
                rag_response = self.answer_query(
                    query=question_text,
                    image_paths=image_paths
                )
                
                generation_time = time.time() - start_time
                
                # Display the RAG response
                self.display_response(rag_response)
                
                # Extract key information from RAG response
                rag_answer = rag_response.get('text', 'No answer generated')
                retrieved_contexts = []
                
                # Extract context information if available
                if 'images' in rag_response:
                    retrieved_contexts = [
                        {
                            'path': img_info.get('path', ''),
                            'metadata': img_info.get('metadata', {}),
                            'relevance_rank': img_info.get('relevance_rank', 0)
                        }
                        for img_info in rag_response['images']
                    ]
                
                # Create result entry
                result_entry = {
                    'question_id': question_id,
                    'question_text': question_text,
                    'context_image_path': context_image_path,
                    'rag_answer': rag_answer,
                    'retrieved_contexts': retrieved_contexts,
                    'generation_metadata': {
                        'generation_time_seconds': generation_time,
                        'timestamp': datetime.now().isoformat(),
                        'source_chunk_id': question_data.get('source_chunk_id', ''),
                        'rag_system_metadata': rag_response.get('metadata', {})
                    }
                }
                
                results.append(result_entry)
                print(f"✅ Answer generated in {generation_time:.2f}s")
                
            except Exception as e:
                print(f"❌ Error processing question {i+1}: {e}")
                # Add error entry to results
                results.append({
                    'question_id': question_data.get('question_id', f'q_{i}'),
                    'question_text': question_data.get('question', ''),
                    'context_image_path': None,
                    'rag_answer': f"Error generating answer: {str(e)}",
                    'retrieved_contexts': [],
                    'generation_metadata': {
                        'error': str(e),
                        'timestamp': datetime.now().isoformat()
                    }
                })
                continue
        
        # Save results
        self._save_results(results)
        
        print(f"\n🎉 RAG Answer Generation Complete!")
        print(f"📊 Processed: {len(results)} questions")
        print(f"✅ Successful: {len([r for r in results if 'error' not in r.get('generation_metadata', {})])}")
        print(f"❌ Errors: {len([r for r in results if 'error' in r.get('generation_metadata', {})])}")
    
    def _save_results(self, results: List[Dict]):
        """
        Save generation results to JSON file.
        
        Args:
            results: List of result dictionaries
        """
        try:
            output_path = Path("/home/students/Leishmania/kaggle/working/rag_generated_answers.json")
            
            # Create comprehensive output structure
            output_data = {
                'metadata': {
                    'generation_timestamp': datetime.now().isoformat(),
                    'total_questions_processed': len(results),
                    'successful_generations': len([r for r in results if 'error' not in r.get('generation_metadata', {})]),
                    'failed_generations': len([r for r in results if 'error' in r.get('generation_metadata', {})]),
                    'rag_system_info': 'Function-based smart_query_system with GPU optimization',
                    'image_directory': str(self.IMAGE_DIR),
                    'chunk_data_sources': len(self.chunk_data_map)
                },
                'generated_answers': results
            }
            
            # Save to file
            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(output_data, f, ensure_ascii=False, indent=2)
            
            print(f"💾 Results saved to: {output_path}")
            print(f"📄 File size: {output_path.stat().st_size / 1024 / 1024:.1f} MB")
            
        except Exception as e:
            print(f"❌ Error saving results: {e}")

# Execute the RAG Answer Generator
# Check if required functions are available from cell 18
required_functions = ['smart_query_system', 'display_multimodal_response']
missing_functions = [func for func in required_functions if func not in globals()]

if not missing_functions:
    print("🔧 Initializing RAG Answer Generator...")
    answer_generator = RAGAnswerGenerator()
    
    print("🚀 Starting answer generation process...")
    answer_generator.run_generation(num_questions_to_run=50)
    
    print("\n✅ Hoàn thành. Kết quả đã được lưu vào: /kaggle/working/rag_generated_answers.json")
    print("🎯 Ready for evaluation phase!")
    
else:
    print(f"❌ Required functions not available: {missing_functions}")
    print("💡 Vui lòng chạy cell 18 (Multimodal RAG System) trước khi chạy cell này.")
    print(f"🔍 Missing functions: {', '.join(missing_functions)}")
    
    # Show available functions for debugging
    available_functions = [name for name in globals() if callable(globals()[name]) and not name.startswith('_')]
    print(f"📋 Available functions: {len(available_functions)} functions found")
    if len(available_functions) < 20:  # Only show if not too many
        print(f"    Functions: {', '.join(available_functions[:10])}{'...' if len(available_functions) > 10 else ''}")